# XiTuner — brand voice gate (Colab)

**The question this notebook answers:** can a small model absorb a brand voice
well enough to beat the same model handed a written style guide in every prompt?

That framing is deliberate, because it is the objection a judge will raise:
brands have style guides, so put the guide in the system prompt and skip
training. For anything a guide can express, that objection is correct.

So the comparison runs the objection at full strength:

| held-out real reply | base + **entire style guide** every call | tuned, **no prompt** |
|---|---|---|

Scores split into two layers:

- **articulable** — rules the guide states outright. The base model *should* do
  well here; that is the control working, not a problem.
- **tacit** — rules nobody wrote down. Never an exclamation mark. Emoji only
  from a fixed set, only in final position. `Aduh` opens a complaint. The
  sign-off appears on complaints and refusals only. No corporate vocabulary.

**The claim lives or dies in the tacit column.** If base-with-guide matches
tuned there, prompting is the honest recommendation and we need to know that
now rather than on camera.

*Nimbus Kopi is a fictional brand. Using a real brand's voice would risk the
contest rule against third-party trademarks in a submission.*

**Before running:** `Runtime → Change runtime type → T4 GPU`.

## 1. Confirm a GPU is attached

Colab silently hands out a CPU runtime when no GPU is free. Gemma 4 E2B has
**5.12B raw parameters** despite the "E2B" (Effective 2B) label — roughly 20.5GB
at float32 — so on CPU it does not merely run slowly, it fails to load.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import subprocess, sys
if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    sys.exit("No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")
print("GPU present")

In [ ]:
%pip install -q -U transformers peft trl datasets accelerate bitsandbytes pydantic python-dotenv

import transformers, peft, trl, datasets
print("transformers", transformers.__version__)
print("peft        ", peft.__version__)
print("trl         ", trl.__version__)
print("datasets    ", datasets.__version__)

## 2. Get the code

Cloned rather than pasted inline, so the notebook runs the same modules as local
development and there is no second copy to keep in sync.

In [ ]:
import os

REPO_URL = "https://github.com/joyedz/xituner.git"

if not os.path.isdir("training"):
    !git clone -q $REPO_URL xituner
    os.chdir("xituner")
print("cwd:", os.getcwd())
assert os.path.isdir("training"), "clone did not land"

os.environ["BASE_MODEL"] = "google/gemma-4-E2B-it"  # ungated: no HF token needed
os.environ.pop("XITUNER_FORCE_CPU", None)

## 3. Build the corpus

350 (incoming message → brand reply) pairs across seven situations, plus 10
held-out pairs cut before anything else and never trained on.

The held-out set is what makes the result checkable by someone who has never
seen this brand: they are not asked whether a reply "sounds right", only which
candidate lands nearer the real one.

In [ ]:
!python -m scripts.make_brand_corpus

## 4. Validate the metrics against the corpus itself

Every reference reply was written to obey all 12 voice rules, so the metrics must
score them 1.00 on both layers. A failure here means the **metric** is wrong, and
a broken metric would quietly invalidate the comparison below.

This check earned its place: it caught four real bugs in the metric (trailing
emoji counted as a third sentence, `"anda"` matching inside `"ganda"`, emoji
demanded on replies that correctly use a sign-off instead, imperatives only
recognised at clause start) and seven replies in the corpus that broke their own
rules.

In [ ]:
!python -m scripts.validate_voice_metrics

## 5. Train

`--load-in-4bit` is required to fit 5.12B raw parameters alongside activations
and gradients in 16GB of T4 VRAM. nf4 brings the weights to roughly 2.6GB.

Note what is absent: no LLM picks the hyperparameters, and no LLM decides when to
stop. Those come from a heuristic table keyed on corpus size and from
`EarlyStoppingCallback`. Gemini's judgment is reserved for corpus surgery,
behavioral refereeing, and diagnosis — work with no deterministic equivalent.

In [ ]:
!python -m training.train_lora \
    --train-file data/brand/train.jsonl \
    --output-dir outputs/nimbus \
    --load-in-4bit

## 6. The three-way comparison

The base model receives the complete style guide on every single call. The tuned
model receives nothing but the incoming message.

The style guide is a genuine, competent brand doc — weakening it on purpose would
rig this and make the result worthless.

In [ ]:
!python -m scripts.compare_voice --adapter-dir outputs/nimbus --load-in-4bit

## 7. Optional: the flawed corpus, and why XiTuner exists

Everything above used a *balanced* corpus. Real brand archives are not balanced:
they are dominated by promo captions and praise, because that is what brands
post. Complaint replies are rare and refusals were never saved at all.

`--flawed` produces that archive: 200 promo, 90 praise, 6 complaints, and **zero
refusals**.

Train on it and the model learns "always be cheerful and promotional", then
answers a refund request with sales energy. That is a brand disaster, and it is a
**data** failure — no learning rate fixes a corpus with no refusals in it.

The held-out set includes refusals precisely so this is measured rather than
asserted. This is the failure XiTuner's Diagnostician exists to catch and
prescribe for.

In [ ]:
!python -m scripts.make_brand_corpus --flawed
!python -m training.train_lora \
    --train-file data/brand/train_flawed.jsonl \
    --output-dir outputs/nimbus_flawed \
    --load-in-4bit
!python -m scripts.compare_voice --adapter-dir outputs/nimbus_flawed --load-in-4bit

## 8. Save before the runtime is recycled

Colab takes local disk with it when the runtime dies. A LoRA adapter is tens of
MB — no reason to lose one.

In [ ]:
import shutil, datetime, os

from google.colab import drive

drive.mount("/content/drive")

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
for name in ("nimbus", "nimbus_flawed"):
    src = f"outputs/{name}"
    if os.path.isdir(src):
        dest = f"/content/drive/MyDrive/xituner/{name}-{stamp}"
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print("saved ->", dest)

## Reading the result honestly

**If tuned wins the tacit column and closeness:** fine-tuning bought something
prompting cannot, measured against replies the model never saw. Build the agent
layer on it.

**If base-with-guide is competitive:** prompting is the honest recommendation for
this task as framed. Do not paper over that by weakening the style guide. Either
the voice needs more genuinely tacit structure, or the corpus needs knowledge the
base model lacks, or the task is the wrong one to build on.

Either way this notebook is development scaffolding. The submission has to show
training on **Vertex AI** driven by an orchestrator on **Cloud Run** — Colab is
not a Google Cloud infrastructure service and does not satisfy that requirement.